# Visualize xarray rasters with GeoLibre

This notebook exercises `Map.add_raster` with both an `xarray.DataArray` and an `xarray.Dataset`. It creates synthetic geographic data, so no download is required.

In [ ]:
# Run once if the raster dependencies are not installed.
# %pip install "geolibre[raster]"

In [ ]:
import numpy as np
import xarray as xr

from geolibre import Map

## DataArray

Coordinate dimensions named `lon` and `lat` are recognized automatically and default to EPSG:4326.

In [ ]:
lon = np.linspace(-125, -65, 240)
lat = np.linspace(50, 24, 140)
xx, yy = np.meshgrid(lon, lat)

temperature = xr.DataArray(
    28 - 0.45 * (yy - 24) + 5 * np.sin((xx + 100) / 8),
    coords={"lat": lat, "lon": lon},
    dims=("lat", "lon"),
    name="temperature",
    attrs={"long_name": "Synthetic air temperature", "units": "°C"},
)
temperature

In [ ]:
m = Map(center=(-96, 37), zoom=3.5, height="700px")
m.add_raster(
    temperature,
    name="Temperature DataArray",
    colormap="turbo",
    rescale=[[-5, 35]],
    array_args={"nodata": -9999, "compress": "LZW"},
)
m

## Dataset variable and dimension selection

Use `array_args["variable"]` to select a data variable and `array_args["isel"]` to select an index from non-spatial dimensions such as time.

In [ ]:
climate = xr.Dataset(
    {
        "temperature": xr.concat([temperature, temperature + 3], dim="time"),
        "precipitation": (
            ("time", "lat", "lon"),
            np.stack(
                [
                    80 + 60 * np.cos((xx + 95) / 10) ** 2,
                    110 + 70 * np.cos((xx + 90) / 10) ** 2,
                ]
            ),
        ),
    },
    coords={"time": ["2026-01-01", "2026-07-01"], "lat": lat, "lon": lon},
)

m.add_raster(
    climate,
    name="Precipitation Dataset",
    colormap="viridis",
    rescale=[[0, 180]],
    array_args={"variable": "precipitation", "isel": {"time": 1}},
)
m

## Multivariable Dataset as RGB

Compatible two-dimensional Dataset variables are written as separate bands. Here the three variables are displayed as red, green, and blue.

In [ ]:
red = np.clip(255 * (xx - lon.min()) / np.ptp(lon), 0, 255).astype("uint8")
green = np.clip(255 * (lat.max() - yy) / np.ptp(lat), 0, 255).astype("uint8")
blue = np.full_like(red, 140)
rgb = xr.Dataset(
    {
        "red": (("lat", "lon"), red),
        "green": (("lat", "lon"), green),
        "blue": (("lat", "lon"), blue),
    },
    coords={"lat": lat, "lon": lon},
)

m.add_raster(
    rgb,
    name="RGB Dataset",
    bands=[1, 2, 3],
    rescale=[[0, 255], [0, 255], [0, 255]],
)
m

The xarray objects are converted to session-scoped temporary Cloud-Optimized GeoTIFFs. Close the widget when finished to remove them.

In [ ]:
# m.close()